## Resumo Executivo

Uma equipe de operações logísticas planeja um ensaio de campo aleatorizado comparando três estratégias de roteirização de motoristas (linha de base estática, reroteirização dinâmica, otimizada por IA) em três regiões de depósito, com o atraso médio de entrega (minutos) como resposta. O PROC GLMPOWER usa um conjunto de dados *exemplar* de médias de célula conjecturadas e resolve o número total de motoristas necessários para detectar os efeitos de estratégia com 80% e 90% de poder, e então mapeia como o poder alcançável se deteriora à medida que a variabilidade de rota para rota cresce.

# Dimensionando um Ensaio de Campo de Roteirização de Motoristas com PROC GLMPOWER

## Resumo Executivo

Uma equipe de operações logísticas está prestes a lançar um ensaio de campo aleatorizado comparando três estratégias de roteirização de motoristas -- uma linha de base **Estática**, um sistema de reroteirização **Dinâmico**, e um planejador **Otimizado por IA** -- implantadas em três regiões de depósito (Nordeste, Meio-Oeste, Oeste). A resposta é o **atraso médio de entrega em minutos**. Antes de comprometer capacidade de frota ao ensaio, a equipe precisa responder: *quantos motoristas precisamos para detectar de forma confiável a melhoria operacional que esperamos?*

Este notebook usa o **PROC GLMPOWER** para realizar uma análise prospectiva de poder e tamanho de amostra para o modelo linear geral por trás do ensaio. Partindo de um conjunto de dados *exemplar* de médias de célula conjecturadas, ele (1) resolve a inscrição total necessária para alcançar 80% e 90% de poder para os efeitos ômnibus de estratégia e região, (2) mapeia como o poder alcançável se degrada à medida que a variabilidade de rota para rota aumenta, e (3) produz uma curva de poder para apoiar a decisão de inscrição.

> **Principal conclusão:** Sob um desvio-padrão do erro plausível de 8 minutos, aproximadamente **63 motoristas** entregam 80% de poder e **83 motoristas** entregam 90% de poder para detectar os efeitos da estratégia de roteirização -- mas se a variabilidade do atraso ficar mais perto de 10 minutos, mesmo 90 motoristas inscritos ficam abaixo de 70% de poder, ressaltando o valor de rotas bem instrumentadas e consistentes.

---
## 1. Construção do Delineamento Exemplar

O primeiro passo codifica o layout do ensaio e o atraso médio *conjecturado* pela equipe para cada combinação estratégia-de-roteirização x região-de-depósito. Fixamos uma semente aleatória e adicionamos uma flutuação (jitter) desprezível para que as médias pareçam realistas enquanto a estrutura balanceada 3x3 é preservada. O peso `cell_n` de 1 em cada célula informa ao GLMPOWER que o delineamento é balanceado.

In [1]:
/* ================================================================
   Gerar o conjunto de dados exemplar de atrasos medios conjecturados.
   Uma linha por celula de delineamento estrategia-de-roteirizacao x
   regiao-de-deposito.
   ================================================================ */
DADOS routing_trial;
   CHAMAR streaminit(20260531);
   COMPRIMENTO routing_strategy $8 depot_region $9;
   VETOR strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   VETOR region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   VETOR smean[3]     _temporary_ (18.0 14.5 11.0);   /* medias de estrategia conjecturadas */
   VETOR radj[3]      _temporary_ (1.5  0.0 -1.0);    /* ajustes regionais (min)  */
   FAZER i = 1 ATÉ 3;
      FAZER j = 1 ATÉ 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         SAÍDA;
      FIM;
   FIM;
   REMOVER i j;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=routing_trial RÓTULO noobs;
   VARIÁVEL routing_strategy depot_region mean_delay_min cell_n;
   RÓTULO routing_strategy="Estratégia de Roteirização" depot_region="Região do Depósito"
         mean_delay_min="Atraso Médio de Entrega (min)" cell_n="Peso da Célula";
   TÍTULO "Médias de Célula Exemplares: Atraso de Entrega Conjecturado (minutos)";
EXECUTAR;

                         Médias de Célula Exemplares: Atraso de Entrega Conjecturado (minutos)                          

   Estratégia de Roteirização    Região do Depósito   Atraso Médio de Entrega (min)   Peso da Célula
Static                         Northeast                               19.687507651                1
Static                         Midwest                                17.9871067398                1
Static                         West                                   16.8240210086                1
Dynamic                        Northeast                              15.9537535365                1
Dynamic                        Midwest                                14.4415169858                1
Dynamic                        West                                   12.8636389493                1
AIOpt                          Northeast                              12.6143811724                1
AIOpt                          Midwest                                


NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Tamanho de Amostra para o Delineamento Ômnibus

Com o delineamento em mãos, pedimos ao GLMPOWER que **resolva o tamanho de amostra total** (`NTOTAL = .`) a 80% e 90% de poder. A instrução `MODEL` especifica um layout de duas vias com interação (`routing_strategy*depot_region`), `WEIGHT cell_n` declara os pesos do perfil balanceado, e `STDDEV = 8` é o erro quadrático médio assumido do atraso de entrega. `NFRACTIONAL` permite que o procedimento reporte contagens fracionárias exatas antes de arredondar para cima.

Também pré-registramos três comparações `CONTRAST` planejadas -- IA-Otimizada vs. Estática, Dinâmica vs. Estática, e qualquer reroteirização vs. Estática -- que documentam as hipóteses operacionalmente relevantes que o ensaio foi construído para testar.

In [2]:
/* ================================================================
   Resolver o total de motoristas necessarios para alcancar 80% e 90%
   de poder para os efeitos de estrategia de roteirizacao e regiao de
   deposito.
   ================================================================ */
PROCEDIMENTO glmpower DADOS=routing_trial;
   CLASSE routing_strategy depot_region;
   MODELO mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   PESO cell_n;
   RÓTULO routing_strategy="Estratégia de Roteirização" depot_region="Região do Depósito"
         mean_delay_min="Atraso Médio de Entrega (min)";
   CONTRAST "IA-Otimizada vs. Estática"           routing_strategy -1  0  1;
   CONTRAST "Dinâmica vs. Estática"                routing_strategy -1  1  0;
   CONTRAST "Qualquer reroteirização vs. Estática" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TÍTULO "Tamanho de Amostra para Detectar Efeitos da Estratégia de Roteirização sobre o Atraso";
EXECUTAR;

                         Médias de Célula Exemplares: Atraso de Entrega Conjecturado (minutos)                          

The GLMPOWER Procedure


                       Fixed Scenario Elements                        

Item                Value                                             
------------------  --------------------------------------------------
Dependent Variable  Atraso Médio de Entrega (min)                     
Source              Estratégia de Roteirização                        
Source              Região do Depósito                                
Source              Estratégia de Roteirização*Região do Depósito     

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Sensibilidade do Poder à Variabilidade e ao Tamanho do Ensaio

A resposta do tamanho de amostra depende do desvio-padrão do erro assumido, que raramente é conhecido com precisão de antemão. Aqui invertemos a pergunta: **fixamos** vários totais de inscrição candidatos (`NTOTAL = 45 90 180`) e **resolvemos o poder alcançado** (`POWER = .`) em uma grade de desvios-padrão de atraso plausíveis (6, 8 e 10 minutos). A tabela resultante é um mapa de sensibilidade -- ela mostra o quão robusto é cada plano de inscrição se a variabilidade real de rota se mostrar maior do que o esperado.

In [3]:
/* ================================================================
   Grade de sensibilidade: poder alcancado atraves de tamanhos de
   ensaio candidatos e desvios-padrao de atraso plausiveis.
   ================================================================ */
PROCEDIMENTO glmpower DADOS=routing_trial;
   CLASSE routing_strategy depot_region;
   MODELO mean_delay_min = routing_strategy depot_region;
   PESO cell_n;
   RÓTULO routing_strategy="Estratégia de Roteirização" depot_region="Região do Depósito"
         mean_delay_min="Atraso Médio de Entrega (min)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TÍTULO "Poder Alcançado Através de Cenários de Variabilidade e Tamanho do Ensaio";
EXECUTAR;

                         Médias de Célula Exemplares: Atraso de Entrega Conjecturado (minutos)                          

The GLMPOWER Procedure


             Fixed Scenario Elements              

Item                Value                         
------------------  ------------------------------
Dependent Variable  Atraso Médio de Entrega (min) 
Source              Estratégia de Roteirização    
Source              Região do Depósito            

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Curva de Poder para a Decisão de Inscrição

Finalmente, traçamos uma **curva de poder** -- poder alcançado à medida que a inscrição total cresce de 30 a 270 motoristas em passos de 30 -- para o modelo estratégia-mais-região no desvio-padrão esperado de 8 minutos. Resolver `POWER = .` através dessa grade de inscrição produz a curva como uma série tabulada de `N Total` vs. `Power`, então podemos ler em qual inscrição cada uma das metas convencionais de 80% e 90% é cruzada.

In [4]:
/* ================================================================
   Curva de poder: poder alcancado vs. total de motoristas inscritos,
   varrido de 30 a 270 em passos de 30 no desvio-padrao esperado de
   8 min.
   ================================================================ */
PROCEDIMENTO glmpower DADOS=routing_trial;
   CLASSE routing_strategy depot_region;
   MODELO mean_delay_min = routing_strategy depot_region;
   PESO cell_n;
   RÓTULO routing_strategy="Estratégia de Roteirização" depot_region="Região do Depósito"
         mean_delay_min="Atraso Médio de Entrega (min)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TÍTULO "Curva de Poder: Poder Alcançado vs. Total de Motoristas Inscritos";
EXECUTAR;

                         Médias de Célula Exemplares: Atraso de Entrega Conjecturado (minutos)                          

The GLMPOWER Procedure


             Fixed Scenario Elements              

Item                Value                         
------------------  ------------------------------
Dependent Variable  Atraso Médio de Entrega (min) 
Source              Estratégia de Roteirização    
Source              Região do Depósito            

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interpretação e Recomendação

A análise dá à equipe de operações um plano de inscrição defensável:

- **Dimensionamento de referência (Seção 2).** Assumindo um desvio-padrão residual de 8 minutos, o modelo completo de duas vias (estratégia, região e sua interação) alcança **80% de poder com 63 motoristas** e **90% de poder com 83 motoristas**. Arredondando para cima por causa de desistências, uma meta de inscrição próxima a **90 motoristas** supera confortavelmente o limiar de 90%.

- **A sensibilidade importa (Seção 3).** O poder é altamente sensível à variabilidade do atraso. Com 90 motoristas, o poder alcançado cai de ~99% (DP=6) para ~87% (DP=8) para ~68% (DP=10). Um piloto de 45 motoristas só é adequado se a variabilidade for baixa (~81% de poder com DP=6) e fica seriamente subdimensionado (~37%) se o DP chegar a 10. A implicação prática: investir em rotas consistentes e bem instrumentadas para manter a variabilidade baixa é tão valioso quanto adicionar motoristas.

- **A curva de poder (Seção 4).** Traçada para o modelo estratégia-mais-região (sem termo de interação, a lente usada para a varredura de sensibilidade), o poder alcançado sobe de 0.37 com 30 motoristas para 0.69 com 60, 0.87 com 90 e 0.95 com 120, achatando acima de 0.99 com 180. Lendo a curva contra as metas, 80% de poder chega perto de 77 motoristas e 90% perto de 99 -- um pouco mais alto do que o dimensionamento do modelo completo na Seção 2, porque remover o termo de interação espalha o mesmo efeito por menos graus de liberdade do modelo.

**Recomendação:** Inscrever aproximadamente **90 motoristas** (30 por estratégia de roteirização, balanceados entre as três regiões de depósito). Isso supera 90% de poder para o modelo completo sob o desvio-padrão esperado de 8 minutos, mantém ~87% de poder mesmo na curva do modelo reduzido mais conservadora, e mantém o ensaio pequeno o suficiente para ser executado dentro de um único trimestre operacional.

*Nota:* o GLMPOWER analisa o *delineamento* conjecturado, não os resultados observados -- então a credibilidade desses números depende das médias e do desvio-padrão conjecturados. As equipes devem revisar o dimensionamento assim que um breve piloto produzir uma estimativa empírica da variabilidade do atraso de entrega.